In [121]:
from pypff import file as PffArchive
from pathlib import Path
import pandas as pd
import gc
import re

In [122]:
pst_dir = Path("../data/raw/pst")
pst_files = list(pst_dir.glob("*.pst"))

print("PST files found:", len(pst_files))
for f in pst_files:
    print(f.name)

PST files found: 14
judy.tse@promab.com (1).pst
judy.tse@promabinc.com(3).pst
archive copy from martyn email by William.pst
martyn.lewis@promab.com by Tim.pst
Judy email backup.pst
judy.tse@promabinc.com(4).pst
archive.pst
Van Inbox backup 05.14.21.pst
martyn.lewis@promab.com (1).pst
antibody.promab@gmail.com.pst
Van sent backup 05.14.21.pst
judy.tse@promab.com (2).pst
judy.tse@promab.com (1)(2).pst
martyn.lewis@promab.com copy by Tim.pst


## **Converting PST to Parquet files**

In [123]:
# =========================
# Basic safe utilities
# =========================
def safe_decode(value):
    """Safely decode bytes / return string / fallback to str(value)."""
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, bytes):
        for enc in ["utf-8", "utf-16", "latin-1"]:
            try:
                return value.decode(enc, errors="ignore")
            except Exception:
                continue
        return str(value)
    try:
        return str(value)
    except Exception:
        return ""

In [124]:
def safe_get(obj, attr, default=None):
    """Safely get an attribute from pypff object."""
    try:
        return getattr(obj, attr)
    except Exception:
        return default

In [125]:
def safe_str(x):
    if x is None:
        return ""
    return str(x)

In [126]:
def extract_header_value(headers, key):
    """
    Extract a header value from raw transport headers.
    Supports folded headers (continuation lines).
    """
    headers = safe_str(headers)
    if not headers.strip():
        return ""

    headers = headers.replace("\r\n", "\n").replace("\r", "\n")
    lines = headers.split("\n")

    collected = []
    collecting = False

    for line in lines:
        if re.match(fr"^{re.escape(key)}\s*:", line, flags=re.IGNORECASE):
            value = re.sub(fr"^{re.escape(key)}\s*:\s*", "", line, flags=re.IGNORECASE)
            collected = [value.strip()]
            collecting = True
        elif collecting and re.match(r"^[ \t]+", line):
            collected.append(line.strip())
        elif collecting:
            break

    return " ".join(collected).strip()

def normalize_message_id(x):
    """
    Normalize message-id style fields for downstream matching.
    Keeps the visible form stable but removes extra spaces/newlines.
    """
    x = safe_str(x).strip()
    if not x:
        return ""

    x = x.replace("\r", " ").replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def extract_message_headers(headers):
    """
    Extract message threading headers from raw transport headers.
    """
    message_id = normalize_message_id(extract_header_value(headers, "Message-ID"))
    in_reply_to = normalize_message_id(extract_header_value(headers, "In-Reply-To"))
    references = normalize_message_id(extract_header_value(headers, "References"))

    return message_id, in_reply_to, references

In [127]:
# =========================
# Header / recipient helpers
# =========================
def extract_email_from_headers(headers, key):
    """
    Extract a single email from raw headers by key, e.g. From / To / Cc.
    Best for fallback use.
    """
    if not headers:
        return ""

    headers = safe_decode(headers)
    key_escaped = re.escape(key)

    # Example: From: John Smith <john@example.com>
    match = re.search(fr"{key_escaped}:.*?<(.*?)>", headers, re.IGNORECASE)
    if match:
        return match.group(1).strip()

    # Example: From: john@example.com
    match = re.search(fr"{key_escaped}:\s*([^\s,;]+@[^\s,;]+)", headers, re.IGNORECASE)
    if match:
        return match.group(1).strip()

    return ""

In [128]:
def extract_recipients(message):
    """Extract all recipient emails from pypff message recipient objects."""
    recipients = []

    try:
        n = safe_get(message, "number_of_recipients", 0) or 0
        for i in range(n):
            try:
                r = message.get_recipient(i)
                email = safe_decode(safe_get(r, "email_address", "")).strip()
                if email:
                    recipients.append(email)
            except Exception:
                continue
    except Exception:
        pass

    return "; ".join(recipients)

In [129]:
def extract_display_names_from_recipients(message):
    """Optional helper: extract recipient display names."""
    names = []
    try:
        n = safe_get(message, "number_of_recipients", 0) or 0
        for i in range(n):
            try:
                r = message.get_recipient(i)
                name = safe_decode(safe_get(r, "name", "")).strip()
                if name:
                    names.append(name)
            except Exception:
                continue
    except Exception:
        pass
    return "; ".join(names)

In [130]:
# =========================
# Field extraction helpers
# =========================
def get_sender_email(msg):
    """
    Robust sender email extraction.
    Priority:
    1. sender_email_address
    2. transport_headers -> From
    3. sender_name if it itself looks like an email
    """
    sender_email = safe_decode(safe_get(msg, "sender_email_address", "")).strip()
    if sender_email:
        return sender_email

    headers = safe_get(msg, "transport_headers", None)
    sender_email = extract_email_from_headers(headers, "From")
    if sender_email:
        return sender_email

    sender_name = safe_decode(safe_get(msg, "sender_name", "")).strip()
    if re.search(r"[^\s]+@[^\s]+", sender_name):
        return sender_name

    return ""

In [131]:
def get_to_field(msg):
    """
    Robust recipient extraction.
    Priority:
    1. display_to
    2. recipient objects
    3. transport_headers -> To
    """
    to_value = safe_decode(safe_get(msg, "display_to", "")).strip()
    if to_value:
        return to_value

    to_value = extract_recipients(msg)
    if to_value:
        return to_value

    headers = safe_get(msg, "transport_headers", None)
    to_value = extract_email_from_headers(headers, "To")
    if to_value:
        return to_value

    return ""

In [132]:
def get_cc_field(msg):
    """Optional cc extraction."""
    cc_value = safe_decode(safe_get(msg, "display_cc", "")).strip()
    if cc_value:
        return cc_value

    headers = safe_get(msg, "transport_headers", None)
    cc_value = extract_email_from_headers(headers, "Cc")
    if cc_value:
        return cc_value

    return ""

In [133]:
def get_body_text(msg):
    """
    Robust body extraction.
    Priority:
    1. plain_text_body
    2. html_body
    """
    body = safe_get(msg, "plain_text_body", None)
    body = safe_decode(body).strip()
    if body:
        return body

    html_body = safe_get(msg, "html_body", None)
    html_body = safe_decode(html_body).strip()
    if html_body:
        return html_body

    return ""

In [134]:
def get_attachment_info(msg):
    """Return (has_attachments, attachment_count, attachment_names)."""
    attachment_names = []
    attachment_count = 0

    try:
        attachment_count = safe_get(msg, "number_of_attachments", 0) or 0
        for k in range(attachment_count):
            try:
                att = msg.get_attachment(k)
                att_name = safe_decode(safe_get(att, "name", "")).strip()
                attachment_names.append(att_name if att_name else None)
            except Exception:
                attachment_names.append(None)
    except Exception:
        attachment_count = 0

    return attachment_count > 0, attachment_count, attachment_names

In [135]:
# =========================
# Main parser
# =========================
def parse_pst_to_records(pst_path):
    """
    Parse one PST file into a list of email records.
    Includes robust fallback logic for sender_email, recipients, body,
    and threading-related headers.
    """
    archive = PffArchive()
    archive.open(str(pst_path))

    records = []
    skipped_messages = 0

    def extract_folder(folder, folder_path=""):
        nonlocal skipped_messages

        current_folder_name = safe_decode(safe_get(folder, "name", "")).strip()
        current_path = f"{folder_path}/{current_folder_name}" if current_folder_name else folder_path

        try:
            n_messages = safe_get(folder, "number_of_sub_messages", 0) or 0
        except Exception:
            n_messages = 0

        for i in range(n_messages):
            try:
                msg = folder.get_sub_message(i)
            except Exception as e:
                skipped_messages += 1
                print(f"  [Skip message {i}] cannot load message | folder={current_path} | error={e}")
                continue

            try:
                subject = safe_decode(safe_get(msg, "subject", "")).strip()
                sender = safe_decode(safe_get(msg, "sender_name", "")).strip()
                sender_email = get_sender_email(msg)
                to_value = get_to_field(msg)
                cc_value = get_cc_field(msg)
                body = get_body_text(msg)

                headers = safe_decode(safe_get(msg, "transport_headers", ""))
                message_id, in_reply_to, references = extract_message_headers(headers)

                recipient_emails = extract_recipients(msg)
                recipient_names = extract_display_names_from_recipients(msg)
                has_attachments, attachment_count, attachment_names = get_attachment_info(msg)

                if not subject:
                    print(f"  [Warning] subject unreadable/empty | folder={current_path} | msg_index={i}")

                record = {
                    "id": safe_get(msg, "identifier", None),
                    "subject": subject,
                    "sender": sender,
                    "sender_email": sender_email,
                    "to": to_value,
                    "cc": cc_value,
                    "date": safe_get(msg, "delivery_time", None),
                    "body": body,
                    "recipient_emails": recipient_emails,
                    "recipient_names": recipient_names,
                    "source_pst": pst_path.name,
                    "folder_path": current_path,
                    "has_attachments": has_attachments,
                    "attachment_count": attachment_count,
                    "attachment_names": attachment_names,
                    "headers": headers,

                    # New fields for alignment / thread reconstruction
                    "message_id": message_id,
                    "in_reply_to": in_reply_to,
                    "references": references,
                    "source_type": "pst",
                    "source_file": str(pst_path.name),
                }

                records.append(record)

            except Exception as e:
                skipped_messages += 1
                print(f"  [Skip bad message {i}] folder={current_path} | error={e}")
                continue

        try:
            n_folders = safe_get(folder, "number_of_sub_folders", 0) or 0
        except Exception:
            n_folders = 0

        for j in range(n_folders):
            try:
                sub_folder = folder.get_sub_folder(j)
                extract_folder(sub_folder, current_path)
            except Exception as e:
                print(f"  [Skip subfolder {j}] parent={current_path} | error={e}")
                continue

    root = archive.get_root_folder()
    extract_folder(root)
    archive.close()

    return records, skipped_messages

In [136]:
# =========================
# Batch process PST files
# =========================
pst_dir = Path("../data/raw/pst")
output_dir = Path("../data/processed/pst_parsed")
output_dir.mkdir(parents=True, exist_ok=True)

pst_files = sorted(pst_dir.glob("*.pst"))
print(f"Found {len(pst_files)} PST files.")

for pst_file in pst_files:
    print(f"\n{'=' * 80}")
    print(f"Processing: {pst_file.name}")

    try:
        records, skipped_messages = parse_pst_to_records(pst_file)
        df = pd.DataFrame(records)

        print("\nPreview:")
        preview_cols = [
            "subject", "sender", "sender_email", "to", "date",
            "message_id", "in_reply_to",
            "has_attachments", "folder_path", "source_file"
        ]
        existing_preview_cols = [c for c in preview_cols if c in df.columns]
        print(df[existing_preview_cols].head(5))

        print("\nSample Body:")
        for i in range(min(3, len(df))):
            print("-" * 60)
            print(f"Subject: {df.loc[i, 'subject']}")
            print(f"Body: {str(df.loc[i, 'body'])[:300]}")

        print("\nMissing / Empty Rates:")
        check_cols = [
            "subject", "sender", "sender_email", "to", "body",
            "message_id", "in_reply_to", "references"
        ]
        for col in check_cols:
            if col in df.columns:
                missing_rate = df[col].isna().mean() if len(df) > 0 else 0
                empty_rate = (df[col].astype(str).str.strip() == "").mean() if len(df) > 0 else 0
                print(f"{col}: missing_rate={missing_rate:.2%}, empty_rate={empty_rate:.2%}")

        print("\nData Info:")
        print(df.info())

        output_file = output_dir / f"{pst_file.stem}.parquet"
        df.to_parquet(output_file, index=False)

        print(f"\nSaved: {output_file.name}")
        print(f"Rows: {len(df)}")
        print(f"Skipped messages: {skipped_messages}")

        del records
        del df
        gc.collect()

    except Exception as e:
        print(f"Failed: {pst_file.name}")
        print("Error:", e)

Found 14 PST files.

Processing: Judy email backup.pst

Preview:
                                             subject    sender sender_email  \
0  Zoom Meeting with Ji Won Ha from Stanford Univ...  Judy Tse                
1    Zoom Meeting With Ji Won at Stanford University  Judy Tse                
2         Declined: ProMab Abcam Bi-Monthly Catch up  Judy Tse                
3                      Accepted: ProMab/ControlPoint  Judy Tse                
4                              FW: Genti:Promab Call  Judy Tse                

  to                date message_id in_reply_to  has_attachments  \
0    2023-02-11 00:39:00                                   False   
1    2023-02-11 00:16:00                                   False   
2    2023-01-09 20:22:00                                   False   
3    2023-01-09 20:18:00                                   False   
4    2022-12-06 20:00:00                                   False   

                            folder_path            

## **Concating Parquet through dataframes**

In [141]:
import pandas as pd

In [142]:
# df = pd.read_parquet("../data/processed/pst_parsed/antibody.promab@gmail.com.parquet")

# print(df.shape)
# df.head()

In [146]:
# 路径
parquet_dir = Path("../data/processed/pst_parsed")
output_path = Path("../data/processed/pst_parsed/parsed_pst_all.parquet")

# 找所有 parquet
files = list(parquet_dir.glob("*.parquet"))

print(f"Total files: {len(files)}")

# 逐个读取 + 拼接
df_pst_all = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

print("Final shape:", df_pst_all.shape)

df_pst_all.to_parquet(output_path, index=False)
print(f"\nSaved master parquet to: {output_path}")

Total files: 13
Final shape: (214803, 21)

Saved master parquet to: ../data/processed/pst_parsed/parsed_pst_all.parquet
